# Bonus: Implementing sklearn estimators

Following the description here: https://scikit-learn.org/stable/developers/develop.html

Source code of scikit-learn is open source (very complicated!): https://github.com/scikit-learn/scikit-learn

Similar tutorial: https://www.geeksforgeeks.org/building-a-custom-estimator-for-scikit-learn-a-comprehensive-guide/

## Preparation

We create a synthetic dataset to test our custom estimator.
The data follows a linear function with some noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# The true coefficients
beta0 = 4
beta1 = 3

# Generate 10 training samples
n = 10
x_low = 0
x_high = 2
X_train = np.random.uniform(low = x_low, high = x_high, size = (n, 1))
y_train = beta0 + beta1 * X_train + np.random.randn(n, 1)

# Use some fixed values as test set
X_test = np.array([0, 1.5, 2]).reshape((-1, 1))
y_test = beta0 + beta1 * X_test


In [ ]:
# Write a function to quickly visualize predictions
def plot_pred(estimator=None, include_true=True):
    # Plot training data
    plt.scatter(X_train, y_train)

    # Plot predictions
    X_test = np.linspace(x_low, x_high, 100).reshape((-1, 1))
    if estimator is not None:
        y_test = estimator.predict(X_test)
        plt.plot(X_test, y_test, 'r-')
    
    # Plot true function
    if include_true:
        plt.plot(X_test, beta0 + beta1 * X_test, 'g--')
    plt.show()

In [ ]:
# Visualize the training data
plot_pred()

## Linear Regression

Use linear regression from sklearn as reference implementation.

In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression(fit_intercept=False)

# Fit the model
lin_reg.fit(X_train, y_train)
print(lin_reg.coef_)

# Predict on the test set
y_pred = lin_reg.predict(X_test)

#  Visualize the results
plot_pred(lin_reg)

Next, we implement our own version of linear regression.
For now, we just implement linear regression without intercept.

In [ ]:
# We somehow need to transform this into an sklearn estimator:
beta = np.linalg.inv(X_train.T @ X_train) @ (X_train.T @ y_train)
y_test = X_test @ beta

In [ ]:
class MyLinReg1():
    def __init__(self):
        # No need to do anything for now
        pass
    
    def fit(self, X_train, y_train):
        # Convert to numpy arrays
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        
        # Compute and store coefficients
        self.beta_ = np.linalg.inv(X_train.T @ X_train) @ (X_train.T @ y_train)

        # Return self (allows chaining)
        return self
        
    def predict(self, X):
        # Convert to numpy array
        X = np.array(X)
        
        # Compute and return prediction
        return X @ self.beta_


We repeat the steps above to test our custom estimator.

In [ ]:
# Instantiate our model
mlin_reg1 = MyLinReg1()

# Fit the model
mlin_reg1.fit(X_train, y_train)
print(mlin_reg1.beta_)

# Predict on the test set
my_pred = mlin_reg1.predict(X_test)

# Visualize the results
plot_pred(mlin_reg1)

Finally, we improve our implementation to include an intercept.

In [ ]:
def addOneColumn(X):
    """
    Helper function to preprend a column of ones to a matrix
    """
    n_columns = X.shape[0]
    one_column = np.ones((n_columns, 1))
    X = np.column_stack([one_column, X])
    return X


from sklearn.base import BaseEstimator, RegressorMixin

class MyLinReg(RegressorMixin, BaseEstimator):
    def __init__(self, fit_intercept = True):
        # Store parameters (nothing to do for now)
        self.fit_intercept = fit_intercept

    def fit(self, X_train, y_train):
        # Convert to numpy arrays
        X_train = np.array(X_train)
        y_train = np.array(y_train)
        
        # Add 1-column for intercept
        if self.fit_intercept:
            X_train = addOneColumn(X_train)
        
        # Compute parameters
        self.beta_ = np.linalg.inv(X_train.T @ X_train) @ (X_train.T @ y_train)

        # Return self (allows chaining)
        return self

    def predict(self, X):
        # Check length of test set
        X = np.array(X)

        # Add 1-column for intercept
        if self.fit_intercept:
            X = addOneColumn(X)
        
        # Compute prediction
        return X @ self.beta_


We test the new implementation again and compare it to sklearn.

In [ ]:
mlin_reg = MyLinReg(True)

# Fit the new model
mlin_reg.fit(X_train, y_train)

# Predict on the test set
y_pred = mlin_reg.predict(X_test)

#  Visualize the results
plot_pred(mlin_reg)

In [ ]:
lin_reg = LinearRegression(fit_intercept=True)

# Fit the model
lin_reg.fit(X_train, y_train)

# Predict on the test set
y_pred = lin_reg.predict(X_test)

#  Visualize the results
plot_pred(lin_reg)

In [ ]:
# Compare coefficients
print("MyLinReg coefficients:", mlin_reg.beta_.flatten())
print("Sklearn coefficients:")
print(lin_reg.intercept_)
print(lin_reg.coef_)

## KNN

In [ ]:
# Our KNN

from sklearn.base import BaseEstimator, RegressorMixin

def getKSmallestIndices(array, k):
    idx = array.argsort()
    idx = idx[0:k]
    return idx

class MyKNeighborsRegressor(RegressorMixin, BaseEstimator):
    def __init__(self, n_neighbors=5):
        # Store parameter (k)
        self.n_neighbors = n_neighbors

    def fit(self, X_train, y_train):
        # Store training data (no computations needed)
        self.X_train = np.array(X_train)
        self.y_train = np.array(y_train)
        
        # Return self (allows chaining)
        return self

    def _predict1(self, x):
        # Check length of training set
        n_train_rows = self.X_train.shape[0]
        
        # Compute distance to each training point
        distances = np.zeros(n_train_rows)
        for j in range(n_train_rows):
            x_j = self.X_train[j,:]
            distances[j] = np.linalg.norm(x - x_j)
        
        # Find k smallest distances == k nearest neighbors
        kNearest = getKSmallestIndices(distances, self.n_neighbors)
        
        # Take mean of k nearest neighbors
        y = np.mean(self.y_train[kNearest])
        return y

    def predict(self, X):
        # Convert to numpy array
        X = np.array(X)

        # Check length of test set
        n_test_rows = X.shape[0]
        
        # Compute y for each test row
        y = np.zeros((n_test_rows, 1))
        for i in range(n_test_rows):
            y[i] = self._predict1(X[i,:])
        
        # Return vector
        return y


In [ ]:
mknn = MyKNeighborsRegressor(2)

mknn.fit(X_train, y_train)
mknn.predict(X_test)

In [ ]:
plot_pred(mknn)

In [ ]:
# sklearn KNN
from sklearn.neighbors import KNeighborsRegressor
knn = KNeighborsRegressor(n_neighbors=2)

knn.fit(X_train, y_train)

knn.predict(X_test)

plot_pred(knn)